# Sample pipeline following Kalibala Abdunoor's work

## Location setup and Imports

In [1]:
import pvlib
# import solarpy
import datetime
import numpy as np
import pandas as pd

In [2]:
# Coordinates for Soroti, Uganda
latitude = 1.7157
longitude = 33.6117
altitude = pvlib.location.lookup_altitude(
    latitude=latitude,
    longitude=longitude
)

# pvlib location object
Soroti = pvlib.location.Location(
    latitude=latitude,
    longitude=longitude,
    altitude=altitude,
    tz='Africa/Kampala'
)

start_date = '2021-01-01'
end_date = '2022-01-01'

days = pd.date_range(start=start_date, end=end_date, freq='D')[:-1]

email="mukiibirogerz@gmail.com"

## Seting up models

In [3]:
#@title Day of the Year
def day_of_the_year(date):
    """
    Returns the day of the year

    Parameters
    ----------
    date : datetime object
        date of interest

    Returns
    -------
    day : int
        day of the year (1 to 365)
    """
    if isinstance(date, datetime.datetime):
        return date.timetuple().tm_yday
    else:
        msg = "date must be a datetime object or array of datetime objects"
        raise TypeError(msg)


In [4]:
Days_of_the_year = [day_of_the_year(day) for day in days]

### Angle of Declination

This is the angular position of the sun at the solar noon (local meridian) with respect to the plane of equator and usually vary between -23.45 and 23.45. Using Cooper equation:

\begin{align}
\delta = 23.45\sin(360\frac{284 + DoY}{365})
\end{align}
where DoY is the day of the year

In [5]:
def declination(day : datetime):
  """
  this function finds solar declination in degrees using Cooper fomrular

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  delta : float
      solar declination in degrees
  """
  day_of_year = day_of_the_year(day)
  delta = 23.45 * np.sin(np.deg2rad(360 * (284 + day_of_year) / 365))
  return delta

### Hour angel (ω)
This is the angular displacement of the sun east or west of local medrian due to rotation of the earth arount its axis.

\begin{align}
ω = 15(solar time - 12)
\end{align}
Where
\begin{align}
solar time = standard time + E + 4(lon)
\end{align}
and
\begin{align}
E = 9.87\cos(2B) - 7.53\sin(B) - 1.5\sin(B)
\end{align}
where B is day angle gven by:
\begin{align}
B = (DoY - 81)\frac{360}{365}
\end{align}

In [6]:
def day_angle(day : datetime):
  """
  this function finds day angle in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  B : float
      day angle in degrees
  """
  day_of_year = day_of_the_year(day)
  B = (day_of_year - 81) * (360 / 365)
  return B

In [7]:
def solar_time(day : datetime):
  """
  this function finds solar time in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  E : float
      solar time in degrees
  """
  day_of_year = day_of_the_year(day)
  B = day_angle(day)
  E = 9.87 * np.cos(np.deg2rad(2 * B)) - 7.53 * np.sin(np.deg2rad(B)) - 1.5 * np.sin(np.deg2rad(B))
  return day.hour + day.minute / 60 + (E + 4*longitude) / 60

In [8]:
def hour_angle(day : datetime):
  """
  this function finds hour angle in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  ω : float
      hour angle in degrees
  """
  return 15 * (solar_time(day) - 12)

### Angle of Elevation (h)

This is the angle between the sun ray tot he obeserver and the projection on the horizontal plane

\begin{align}
  \sin(h) = \sin(\delta)\sin(Φ) + \cos(δ)\cos(Φ)\cos(ω)
\end{align}

where
* h is angle of elevation
* δ is angle of declination
* Φ is latitude
* ω is hour angle

In [9]:
def angle_of_elevation(day : datetime, latitude : float):
  """
  this function finds angle of elevation in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  h : float
      angle of elevation in degrees
  """
  delta = declination(day)
  hour_angle = hour_angle(day)

  h = np.arcsin(np.sin(np.deg2rad(delta)) * np.sin(np.deg2rad(latitude)) +
                np.cos(np.deg2rad(delta)) * np.sin(np.deg2rad(latitude)) *
                np.cos(np.deg2rad(hour_angle)))

  return np.rad2deg(h)

### Solar azimuth (γ)
This the angular displacement from south of the projection beam radiation on horizontal plane

\begin{align}
  \cos(γ) = \frac{\sin(h)\sin(Φ) - \sin(δ)}{\cos(h)\cos(Φ)}
\end{align}


In [10]:
def solar_azimuth(day : datetime, latitude : float):
  """
  this function finds solar azimuth in degrees

  Parameters
  ----------
  day : datetime object
      date of interest
  latitude : float
      latitude of interest

  Returns
  -------
  azimuth : float
      solar azimuth in degrees
  """
  delta = declination(day)
  h = angle_of_elevation(day, latitude)

  azimuth = np.arccos((np.sin(np.deg2rad(h)) * np.sin(np.deg2rad(latitude)) -
                np.sin(np.deg2rad(delta))) / (np.cos(np.deg2rad(h)) *
                np.cos(np.deg2rad(latitude))))

  return np.rad2deg(azimuth)

### Angle of incidence

This is the angle between the sun ray on the surface and the normal to the plane of incidennce

\begin{align}
  \sin(ν) = \sin(δ)\sin(Φ)\cos(α) - \sin(δ)\cos(Φ)\sin(α)\cos(β) + \cos(δ)\cos(Φ)\cos(α)\cos(ω) + \cos(δ)\sin(Φ)\sin(α)\cos(β)\cos(ω) + \cos(δ)\sin(α)\sin(β)\sin(ω)
\end{align}

where:
* ν is angle of incidence
* α is surface tilt
* β is surface azimuth

In [11]:
def angle_of_incidence(day : datetime, latitude : float, surface_azimuth : float, surface_tilt : float):
  """
  this function finds angle of incidence in degrees

  Parameters
  ----------
  day : datetime object
      date of interest
  latitude : float
      latitude of interest
  surface_azimuth : float
      surface azimuth in degrees
  surface_tilt : float
      surface tilt in degrees

  Returns
  -------
  aoi : float
      angle of incidence in degrees
  """
  delta = declination(day)
  hour_angle = hour_angle(day)

  aoi = np.arcsin(np.sin(np.deg2rad(delta)) * np.sin(np.deg2rad(latitude)) * np.cos(np.deg2rad(surface_tilt)) -
                np.sin(np.deg2rad(delta)) * np.cos(np.deg2rad(latitude)) * np.sin(np.deg2rad(surface_tilt)) * np.cos(np.deg2rad(surface_azimuth)) +
                np.cos(np.deg2rad(delta)) * np.cos(np.deg2rad(latitude)) * np.cos(np.deg2rad(surface_tilt)) * np.cos(np.deg2rad(hour_angle)) +
                np.cos(np.deg2rad(delta)) * np.sin(np.deg2rad(latitude)) * np.sin(np.deg2rad(surface_tilt)) * np.cos(np.deg2rad(surface_azimuth)) * np.cos(np.deg2rad(hour_angle)) +
                np.cos(np.deg2rad(delta)) * np.sin(np.deg2rad(surface_azimuth)) * np.sin(np.deg2rad(surface_tilt)) * np.sin(np.deg2rad(hour_angle)))

  return np.rad2deg(aoi)


## Extraterrestrial Radiation Models

### Duffie and Beckman

\begin{align}
  G_{ET,h} = I_o(1 + 0.033\cos(\frac{360*DOY}{365}))\sin(h)
\end{align}

Where
\begin{align}
  I_o = 1367 W/m^2
\end{align}
### Spencer

\begin{align}
  G_{ET,h} = I_o(1.000110 + 0.034221\cos(B^*) + 0.001280\sin(B^*) + 0.000719\cos(2B^*) + 0.000077\sin(2B^*))\sin(h)
\end{align}

where
\begin{align}
  B^* = (DoY - 1)\frac{360}{365}
\end{align}

### ASCE

\begin{align}
  G_{ET,h} = I_o(1 + 0.033\cos(B))\sin(h)
\end{align}


In [12]:
def B_star(day : datetime):
  """
  this function finds B* in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  B_star : float
      B* in degrees
  """

  day_of_year = day_of_the_year(day)
  B_star = (day_of_year - 1) * (360 / 365)
  return B_star

In [13]:
def extra_irr_h(day : datetime, latitude : float):
  """
  this function finds extraterrestrial radiation in degrees

  Parameters
  ----------
  day : datetime object
      date of interest
  latitude : float
      latitude of interest

  Returns
  -------
  np array of G_ET_h values

  """

  solar_constant = 1367
  DoY = day_of_the_year(day)
  B_radians = np.deg2rad(B_star(day))
  h = angle_of_elevation(day, latitude)


  E_spencer = solar_constant * (1.000110 + 0.034221 * np.cos(B_radians) + 0.00128 * np.sin(B_radians) + 0.000719 * np.cos(2 * B_radians) + 0.000077 * np.sin(2 * B_radians)) * np.sin(h)

  E_asce = solar_constant * (1 + 0.033 * np.cos(B_radians)) * np.sin(h)

  E_duffie = solar_constant * (1 + 0.033 * np.cos(np.deg2rad(DoY*(360/365)))) * np.sin(h)

  return np.array([x for x in [E_spencer, E_asce, E_duffie]])

In [14]:
def extra_irr(day : datetime, latitude : float):
  """
  this function finds extraterrestrial radiation in degrees

  Parameters
  ----------
  day : datetime object
      date of interest
  latitude : float
      latitude of interest

  Returns
  -------
  np array of G_ET_h values

  """

  solar_constant = 1367
  DoY = day_of_the_year(day)
  B_radians = np.deg2rad(B_star(day))
  h = angle_of_elevation(day, latitude)


  E_spencer = solar_constant * (1.000110 + 0.034221 * np.cos(B_radians) + 0.00128 * np.sin(B_radians) + 0.000719 * np.cos(2 * B_radians) + 0.000077 * np.sin(2 * B_radians))

  E_asce = solar_constant * (1 + 0.033 * np.cos(B_radians))

  E_duffie = solar_constant * (1 + 0.033 * np.cos(np.deg2rad(DoY*(360/365))))

  return np.array([x for x in [E_spencer, E_asce, E_duffie]])

## Decomposition Model

Decomposition model estimates the fraction beam and diffuse horizontal irradiance from measured global horizontal irradiance. The cardinal input of these models are GHI and clearnes index kt. Clearness index is the function which compute global horizontal clearness index with respect to the extraterestrial irradiance on horizontal plane

\begin{align}
  K_t = \frac{G_h}{G_{ET,h}}
\end{align}

Where Kd is diffuse fraction

\begin{align}
  K_d = \frac{G_{d, h}}{G_h}
\end{align}


### Muneer

\begin{align}
f_d(K_d) =
\begin{cases}
0.95 & \text{if } K_t < 0.175, \\
0.9698 + 0.4353K_t - 3.4499K_t^2 + 2.1888K_t^3 & \text{if } 0.175 < K_t \leq 0.755, \\
0.26 & \text{if } K_t > 0.755
\end{cases}
\end{align}



### Noor2


### Erbs

\begin{align}
  G_h = G_{b,h} + G_{d,h}
\end{align}

\begin{align}
f_d(K_d) =
\begin{cases}
1.0 - 0.09 K_t & \text{if } K_t \leq 0.22, \\
0.9511 - 0.1604K_t + 4.388K_t^2 - 16.638K_t^3 + 12.336K_t^4 & \text{if } 0.22 < K_t \leq 0.8, \\
0.165 & \text{if } K_t > 0.8
\end{cases}
\end{align}

In [15]:
def erbs(time : datetime, latitude : float, GHI : float):
  """
  this function finds extraterrestrial radiation in degrees

  Parameters
  ----------
  time : datetime object
      date of interest
      latitude : float
      latitude of interest
  GHI : float
      global horizontal irradiance

  Returns
  -------
  np array of G_ET_h values

  """

  Extr = extra_irr(time, latitude)
  kt = GHI / Extr
  kt = np.stack(kt.values)
  kt = np.maximum(kt, 0)

  k_d = np.where(kt <= 0.22, 1.0 - 0.09 * kt, np.where((kt > 0.22) & (kt <= 0.8), 0.9511 - 0.1604 * kt + 4.388 * kt**2 - 16.638 * kt**3 + 12.336 * kt**4, 0.165))

  GHI = GHI[:, np.newaxis]
  BHI = GHI * (1 - k_d)
  DHI = GHI - BHI

  return np.array([x for x in [BHI, DHI]])

### Orgil and Hollands
\begin{align}
  G_h = G_{b,h} + G_{d,h}
\end{align}

\begin{align}
f_d(K_d) =
\begin{cases}
1.0 - 0.249 K_t & \text{if } K_t < 0.35, \\
1.577 - 1.84K_t & \text{if } 0.35 < K_t \leq 0.75, \\
0.177 & \text{if } K_t > 0.75
\end{cases}
\end{align}


In [16]:
def orgil_hollands(time : datetime, latitude : float, GHI : float):
  """
  this function finds extraterrestrial radiation in degrees

  Parameters
  ----------
  time : datetime object
      date of interest
      latitude : float
      latitude of interest
  GHI : float
      global horizontal irradiance

  Returns
  -------
  np array of G_ET_h values
  """

  Extr = extra_irr(time, latitude)
  kt = GHI / Extr
  kt = np.stack(kt.values)
  kt = np.maximum(kt, 0)

  k_d = np.where(kt <= 0.35, 1.0 - 0.249 * kt, np.where((kt >= 0.35) & (kt <= 0.75), 1.577 - 1.84 * kt, 0.177))

  GHI = GHI[:, np.newaxis]
  BHI = GHI * (1 - k_d)
  DHI = GHI - BHI

  return np.array([x for x in [BHI, DHI]])

### Boland

\begin{align}
  K_d = \frac{1}{1 + e^{7.997(K_t - 0.587)}}
\end{align}


In [17]:
def boland(time : datetime, latitude : float, GHI : float):
  """
  this function finds extraterrestrial radiation in degrees

  Parameters
  ----------
  time : datetime object
      date of interest
      latitude : float
      latitude of interest
  GHI : float
      global horizontal irradiance
  Returns
  -------
  np array of G_ET_h values
  """

  Extr = extra_irr(time, latitude)
  kt = GHI / Extr
  kt = np

  kt = np.where(kt < 0, 0.587, kt)
  den = 1 + np.exp(7.997 * (kt - 0.587))
  k_d = 1 / den

  GHI = GHI[:, np.newaxis]
  BHI = GHI * (1 - k_d)
  DHI = GHI - BHI

  return np.array([x for x in [BHI, DHI]])

### Noor1

\begin{align}
k_d = 0.2522 + 4.5394k_t + -15.2958 k_t^2 + 17.32080k_t^3 - 6.4097k_t^4
\end{align}
for
\begin{align}
 0 \leq k_t \leq 1
\end{align}

In [18]:
def Noor1(time : datetime, latitude : float, GHI : float):
  """
  this function finds extraterrestrial radiation in degrees

  Parameters
  ----------
  time : datetime object
      date of interest
      latitude : float
      latitude of interest
  GHI : float
      global horizontal irradiance

  Returns
  -------
  np array of G_ET_h values
  """

  Extr = extra_irr(time, latitude)
  kt = GHI / Extr
  kt = np.stack(kt.values)
  kt = np.maximum(kt, 0)

  k_d = np.where((kt >= 0) & (kt <= 1) , 0.2522 + 4.5394 * kt + -15.2958 * kt**2 + 17.32080 * kt**3 - 6.4097 * kt**4, np.nan)

  GHI = GHI[:, np.newaxis]
  BHI = GHI * (1 - k_d)
  DHI = GHI - BHI

  return np.array([x for x in [BHI, DHI]])

### Noor2

\begin{align}
f_d(K_d) =
\begin{cases}
-0.0925 + 32.7710k_t - 699.9121k_t^2 + 6578.6933k_t^3 - 30060.3914k_t^4 + 51585.2035 k_t^5 & \text{if } K_t < 0.2, \\
0.5778 + 1.3075K_t - 4.7547K_t^2 + 3.4303K_t^3 & \text{if } 0.2 \leq K_t < 0.8, \\
0.85329 - 1.64520k_t + 1.22167k_t^2 & \text{if } K_t \geq 0.8
\end{cases}
\end{align}

In [ ]:
def Noor2(time : datetime, latitude : float, GHI : float):
  """
  this function finds extraterrestrial radiation in degrees

  Parameters
  ----------
  time : datetime object
      date of interest
      latitude : float
      latitude of interest
  GHI : float
      global horizontal irradiance

  Returns
  -------
  np array of G_ET_h values
  """

  Extr = extra_irr(time, latitude)
  kt = GHI / Extr
  kt = np.stack(kt.values)
  kt = np.maximum(kt, 0)

  k_d = np.where( kt < 0.2, -0.0925 + 32.7710 * kt - 699.9121 * kt**2 + 6578.6933 * kt**3 - 30060.3914 * kt**4 + 51585.2035 * kt**5,
                 np.where((kt >= 0.2) & (kt < 0.8), 0.5778 + 1.3075 * kt - 4.7547 * kt**2 + 3.4303 * kt**3,
                          np.where((kt >= 0.8) & (kt <= 1), 0.85329 - 1.64520 * kt + 1.22167 * kt**2, np.nan)))

  GHI = GHI[:, np.newaxis]
  BHI = GHI * (1 - k_d)
  DHI = GHI - BHI

  return np.array([x for x in [BHI, DHI]])

## Transposition Model
- Lui and Jordan
- Perez
- Hay and Davies
- Klucher
- Reindl et al